# Naive RAG Pipeline

The **naive RAG pipeline** is the minimum viable document-grounded assistant: **index** a corpus into a vector store, **retrieve** top-$k$ chunks for each user question, **augment** the LLM prompt with that evidence, and **generate** a chat completion. No reranking, no query rewriting, no agents, no hybrid search — just the four-step loop that every advanced RAG pattern extends.

Notebook **01** explained *why* retrieval matters before generation. This notebook implements the *how*: a working end-to-end path with **in-memory Chroma**, OpenAI embeddings (`text-embedding-3-small`), and `gpt-4o-mini` on the shared FastAPI Notes API corpus (chunks **c1–c5**).

You will index documents, run retrieval for auth / migration / pytest questions, build prompts, compare bare LLM answers against RAG answers, inspect token usage, and wrap the flow in a reusable `naive_rag` helper returning structured results.

Prerequisites: **01. Introduction to RAG**, **05. Embeddings & Vector DBs** (especially **03** embedding APIs, **04** chunking, **06** Chroma). Next: **03. RAG Architecture Patterns**.


In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import chromadb
from dataclasses import dataclass
from openai import OpenAI

openai_client = OpenAI(api_key=OPENAI_API_KEY)
EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4o-mini"

DOCUMENTS = [
    {
        "id": "c1",
        "text": (
            "The Notes API is built with FastAPI and stores notes in memory for demos. "
            "Routes live under /notes with GET, POST, PUT, and DELETE handlers."
        ),
        "source": "api-docs",
    },
    {
        "id": "c2",
        "text": (
            "Alembic applies SQLAlchemy schema migrations. "
            "Run alembic upgrade head after pulling new revisions."
        ),
        "source": "db-docs",
    },
    {
        "id": "c3",
        "text": (
            "JWT bearer tokens authenticate API requests. "
            "Clients send Authorization: Bearer <token> header."
        ),
        "source": "security-docs",
    },
    {
        "id": "c4",
        "text": (
            "Pytest fixtures share database setup across tests. "
            "Use conftest.py for session-scoped engines and dependency overrides."
        ),
        "source": "test-docs",
    },
    {
        "id": "c5",
        "text": (
            "Alembic autogenerate compares SQLAlchemy models to the live database schema "
            "and drafts revision files."
        ),
        "source": "db-docs",
    },
]


def embed_texts(texts: list[str]) -> list[list[float]]:
    response = openai_client.embeddings.create(model=EMBED_MODEL, input=texts)
    ordered = sorted(response.data, key=lambda item: item.index)
    return [item.embedding for item in ordered]


def fresh_collection(name: str = "naive_rag_lesson"):
    client = chromadb.Client()
    try:
        client.delete_collection(name)
    except Exception:
        pass
    return client.create_collection(name=name, metadata={"hnsw:space": "cosine"})


print("Setup OK —", len(DOCUMENTS), "chunks, embed model:", EMBED_MODEL)


---

## 1. What Is a Naive RAG Pipeline?

**Naive RAG** (sometimes called *vanilla RAG* or *baseline RAG*) is the simplest architecture that still deserves the name:

1. **Index** — embed each document chunk once; store vectors + text + metadata in Chroma.
2. **Retrieve** — embed the user question; fetch top-$k$ neighbors by cosine distance.
3. **Augment** — insert retrieved passages into a prompt with grounding instructions.
4. **Generate** — call a chat model; return the completion to the user.

At a high level:

$$\text{answer} = \text{LLM}\big(\text{prompt}(q,\; \text{Retrieve}(q,\; \mathcal{D}))\big)$$

where $q$ is the user query and $\mathcal{D}$ is the indexed chunk collection.

### 1.1 Why Start Here?

| Reason | Detail |
|--------|--------|
| **Debuggable** | Each stage has inspectable inputs and outputs |
| **Composable** | Later notebooks swap retrieval, prompting, or generation |
| **Production-shaped** | Real assistants still run this loop under the hood |
| **Curriculum anchor** | Notebooks **03–10** extend what naive RAG omits |

Naive RAG is not *wrong* — it is the **baseline** you measure improvements against.


### 1.2 Offline vs Online Phases

RAG systems split into two phases with different latency and cost profiles:

```
OFFLINE (batch, run on ingest)          ONLINE (per user query)
──────────────────────────────          ─────────────────────────
┌──────────┐                            ┌──────────┐
│ documents│                            │ question │
└────┬─────┘                            └────┬─────┘
     │ chunk (already done here)              │ embed query
     ▼                                        ▼
┌──────────┐     ┌──────────┐            ┌──────────┐
│  embed   │────►│  Chroma  │◄───────────│  query   │
│  (API)   │     │  index   │  top-k     │  (k=3)   │
└──────────┘     └──────────┘            └────┬─────┘
                                              │
                                              ▼
                                    ┌─────────────────┐
                                    │ prompt assembly │
                                    │ context + rules │
                                    └────────┬────────┘
                                             │
                                             ▼
                                    ┌─────────────────┐
                                    │ chat completion │
                                    │  (gpt-4o-mini)  │
                                    └─────────────────┘
```

**Offline** work is amortized: you pay embedding cost once per chunk. **Online** work repeats per question: one query embedding + one (sometimes large) chat call.


---

## 2. The Shared Corpus (c1–c5)

This RAG section reuses a tiny **FastAPI Notes API** knowledge base so you can compare retrieval behavior across notebooks without re-learning a new domain.

| id | source | Topic |
|----|--------|-------|
| **c1** | api-docs | FastAPI Notes API routes |
| **c2** | db-docs | Alembic `upgrade head` |
| **c3** | security-docs | JWT bearer authentication |
| **c4** | test-docs | Pytest fixtures / conftest |
| **c5** | db-docs | Alembic autogenerate |

### 2.1 Why Metadata Matters

Each chunk carries a `source` tag. At generation time you can:

- Ask the model to **cite** which document family it used
- **Filter** retrieval later (notebook **07**)
- **Debug** wrong answers by tracing chunk provenance

In production, metadata often includes `page`, `section`, `version`, `updated_at`, and `acl` — here we keep one field to stay focused on the pipeline.


In [ ]:
for doc in DOCUMENTS:
    src = doc["source"]
    print(doc["id"], "|", src)
    print("  ", doc["text"][:72], "...")
    print()


---

## 3. Step 1 — Index the Corpus

Indexing is the offline phase. For each chunk we:

1. Call OpenAI `embeddings.create` with `text-embedding-3-small`
2. Store `ids`, `documents`, `embeddings`, and `metadatas` in a Chroma collection
3. Set `metadata={"hnsw:space": "cosine"}` so distance matches our embedding workflow

| Field | Purpose |
|-------|---------|
| `ids` | Stable chunk identifier (`c1`…`c5`) |
| `documents` | Raw text returned at query time |
| `embeddings` | 1536-dim vectors from OpenAI |
| `metadatas` | Filterable tags (`source`) |

We use `chromadb.Client()` (in-memory). Data is **lost** when the kernel restarts — fine for learning. Notebook **05.06** also showed `PersistentClient` for durability.


In [ ]:
collection = fresh_collection("naive_rag_demo")

texts = [d["text"] for d in DOCUMENTS]
ids = [d["id"] for d in DOCUMENTS]
metas = [{"source": d["source"]} for d in DOCUMENTS]
vectors = embed_texts(texts)

collection.add(
    ids=ids,
    documents=texts,
    embeddings=vectors,
    metadatas=metas,
)

print("Indexed", collection.count(), "chunks")
print("Embedding dimension:", len(vectors[0]))


**Important:** use the **same** `EMBED_MODEL` at index and query time. Mixing models (e.g. `text-embedding-3-small` at index, `text-embedding-3-large` at query) breaks vector geometry — retrieved chunks will look random.

Re-index when you change chunking, embedding model, or document text. Version your index name (`docs_v3`) in production so you can roll back.


---

## 4. Step 2 — Retrieve Relevant Chunks

Given a natural-language question $q$:

1. Embed $q$ with the same model used at index time
2. Call `collection.query(query_embeddings=[q_vec], n_results=k)`
3. Inspect `documents`, `metadatas`, and `distances`

### 4.1 The `retrieve` Function

Keep retrieval as a **pure function** of `(query, k)` so you can unit-test it without calling the LLM.


In [ ]:
def retrieve(query: str, k: int = 3):
    q_vec = embed_texts([query])[0]
    return collection.query(
        query_embeddings=[q_vec],
        n_results=k,
        include=["documents", "metadatas", "distances", "ids"],
    )


def print_hits(label: str, hits, k: int):
    print(label)
    docs = hits["documents"][0]
    metas = hits["metadatas"][0]
    dists = hits["distances"][0]
    chunk_ids = hits["ids"][0]
    for rank, (cid, doc, meta, dist) in enumerate(zip(chunk_ids, docs, metas, dists), start=1):
        src = meta["source"]
        print(f"  [{rank}] id={cid}  dist={dist:.4f}  source={src}")
        print("      ", doc)
    print()


In [ ]:
AUTH_QUESTION = "How do clients authenticate to the Notes API?"

auth_hits = retrieve(AUTH_QUESTION, k=3)
print_hits("Q: " + AUTH_QUESTION, auth_hits, k=3)


### 4.2 Distance vs Similarity (Chroma + Cosine)

Chroma returns **distances**, not similarities. With `hnsw:space: cosine`:

| Metric | Direction | Formula (intuition) |
|--------|-----------|---------------------|
| **Cosine distance** (Chroma) | Lower = closer | $1 - \cos(\theta)$ for normalized vectors |
| **Cosine similarity** (NumPy) | Higher = closer | $\cos(\theta) = \frac{a \cdot b}{\|a\|\|b\|}$ |

```
  distance = 0.00  ──► identical direction (best match)
  distance = 0.30  ──► strong semantic match (typical top-1)
  distance = 0.80  ──► weak / off-topic
  distance = 1.00  ──► orthogonal (unrelated)
```

When you see `distance=0.21` on chunk **c3** for an auth question, that is a **good** sign — not a percentage confidence score. Calibrate thresholds on **your** corpus; do not treat distance as probability.

Notebook **05.02** covered NumPy similarity; notebook **05.06** aligned Chroma's cosine space with OpenAI embeddings.


---

## 5. Choosing $k$ (How Many Chunks to Retrieve)

`n_results=k` is the simplest hyperparameter in RAG — and one of the most impactful.

| $k$ | Trade-off |
|-----|-----------|
| **1** | Precise but fragile — miss multi-hop evidence |
| **3** | Good default for small corpora (5–50 chunks) |
| **5–10** | Useful when answers span several sections |
| **20+** | Risks context bloat, higher cost, confused model |

### 5.1 Heuristics for This Corpus

With only **five** chunks, $k=3$ usually returns the right topic plus one neighbor. Auth questions should rank **c3** first; migration questions should rank **c2** and **c5**.

**Rule of thumb:** start with $k=3$, log retrieved ids per query, and increase $k$ only when you see **missing** evidence in the top results — not when the model hallucinates (that is often a generation or prompt issue).


In [ ]:
MIGRATION_QUESTION = "What command applies Alembic migrations after pulling code?"

for k in (1, 3, 5):
    hits = retrieve(MIGRATION_QUESTION, k=k)
    ids_row = hits["ids"][0]
    dists = hits["distances"][0]
    pairs = ", ".join(f"{cid}({d:.3f})" for cid, d in zip(ids_row, dists))
    print(f"k={k}: {pairs}")


---

## 6. Step 3 — Build Context from Retrieved Chunks

Retrieval returns a **list** of passages. The LLM needs a **single string** with clear boundaries. A minimal pattern:

```
[source=security-docs]
JWT bearer tokens authenticate API requests...

[source=api-docs]
The Notes API is built with FastAPI...
```

Delimiters (`---`, XML tags, `[CHUNK id=c3]`) help the model separate **evidence** from **instructions**. Notebook **05** goes deeper on citation formats and refusal behavior.


In [ ]:
def build_context(hits) -> str:
    blocks = []
    docs = hits["documents"][0]
    metas = hits["metadatas"][0]
    chunk_ids = hits["ids"][0]
    for cid, doc, meta in zip(chunk_ids, docs, metas):
        src = meta["source"]
        block = f"[id={cid} source={src}]\n{doc}"
        blocks.append(block)
    return "\n\n".join(blocks)


context_preview = build_context(auth_hits)
print(context_preview[:400], "...")


---

## 7. System vs User Message Design

Chat models read a **list of messages**, not one flat string. Naive RAG must decide what goes where.

| Content | Typical role | Why |
|---------|--------------|-----|
| Persona, rules, refusal policy | **system** | Stable across turns; high authority |
| Retrieved evidence + user question | **user** | Changes every request |

### 7.1 A Factual Q&A Template

**System message** (stable):

> You are a helpful assistant for the FastAPI Notes API project. Answer ONLY using the provided context. If the context does not contain the answer, say you do not know. Cite chunk id and source when possible.

**User message** (per query):

```
Context:
[retrieved blocks]

Question: {user_question}
```

### 7.2 Design Choices

| Choice | Recommendation for factual docs |
|--------|------------------------------|
| `temperature` | **0** — reduces creative drift off-context |
| `max_tokens` | Cap completion length for support bots |
| Context placement | User message (easy to log and swap) |
| Streaming | Better UX — covered in architecture notebook **03** |

Putting context in the **system** message works for some models but can be harder to audit in logs. For this curriculum we use **system = rules**, **user = context + question**.


In [ ]:
SYSTEM_MESSAGE = (
    "You are a helpful assistant for the FastAPI Notes API project. "
    "Answer ONLY from the provided context. "
    "If the answer is not in the context, say you do not know. "
    "Cite chunk id and source tag when possible."
)


def build_user_message(question: str, context: str) -> str:
    return "Context:\n" + context + "\n\nQuestion: " + question


user_msg = build_user_message(AUTH_QUESTION, context_preview)
print("System (first 80 chars):", SYSTEM_MESSAGE[:80], "...")
print("User message length (chars):", len(user_msg))


---

## 8. Step 4 — Generate the Answer

The generator calls `chat.completions.create`. The model **reads** context at inference time — it does **not** update weights. That is what makes RAG cheap to refresh: change the index, not the model.

| Parameter | Value here | Role |
|-----------|------------|------|
| `model` | `gpt-4o-mini` | Fast, inexpensive chat model |
| `temperature` | `0` | Deterministic, factual tone |
| `messages` | system + user | Grounded completion |


In [ ]:
def generate(system: str, user: str) -> tuple[str, object]:
    response = openai_client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
    )
    answer = response.choices[0].message.content
    return answer, response


answer_auth, resp_auth = generate(SYSTEM_MESSAGE, user_msg)
print("Question:", AUTH_QUESTION)
print("\nAnswer:\n", answer_auth)


---

## 9. Bare LLM vs RAG — Why Retrieval Matters

A bare LLM call sends **only** the question (plus a generic system prompt). The model answers from **parametric memory** — patterns learned during training — which may be wrong for *your* project specifics.

RAG injects **non-parametric memory**: the actual c1–c5 text. Compare the two on the same question.


In [ ]:
bare_response = openai_client.chat.completions.create(
    model=CHAT_MODEL,
    temperature=0,
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant for the FastAPI Notes API project.",
        },
        {"role": "user", "content": AUTH_QUESTION},
    ],
)
bare_answer = bare_response.choices[0].message.content
print("=== BARE LLM (no retrieval) ===")
print(bare_answer)


In [ ]:
print("=== RAG (with retrieved context) ===")
print(answer_auth)
print()
print("Retrieved chunk ids:", auth_hits["ids"][0])


The bare answer may sound plausible (OAuth2, API keys, session cookies) but only **c3** states the project's actual rule: **JWT bearer tokens** in the `Authorization` header. RAG does not guarantee truth — but it **anchors** the model to your corpus.

When the bare answer happens to be correct, it is because training data overlapped your docs — not because the system is **grounded**.


---

## 10. Token Usage Inspection

RAG cost is dominated by **input tokens** (context + question). Always log usage in development.

| Field | Meaning |
|-------|---------|
| `prompt_tokens` | System + user input |
| `completion_tokens` | Generated answer |
| `total_tokens` | Sum billed for the call |

Embedding calls have separate usage on the embeddings response. Budget notebooks: context stuffing grows linearly with $k$ and chunk size (see **06. Context Assembly**).


In [ ]:
usage = resp_auth.usage
print("prompt_tokens:    ", usage.prompt_tokens)
print("completion_tokens:", usage.completion_tokens)
print("total_tokens:     ", usage.total_tokens)

embed_resp = openai_client.embeddings.create(model=EMBED_MODEL, input=[AUTH_QUESTION])
embed_usage = embed_resp.usage
print("\nEmbedding total_tokens:", embed_usage.total_tokens)


---

## 11. Structured Results with `RAGResult`

Wrap pipeline output in a dataclass (or typed dict) so callers get **answer**, **chunk_ids**, and **context** for logging, UI citations, and eval harnesses (notebook **09**).

| Field | Use |
|-------|-----|
| `answer` | Show to user |
| `chunk_ids` | Audit trail / click-through |
| `context` | Replay exact evidence sent to the model |


In [ ]:
@dataclass
class RAGResult:
    answer: str
    chunk_ids: list[str]
    context: str


def naive_rag(question: str, k: int = 3) -> RAGResult:
    hits = retrieve(question, k=k)
    ctx = build_context(hits)
    user = build_user_message(question, ctx)
    ans, _raw = generate(SYSTEM_MESSAGE, user)
    ids_used = list(hits["ids"][0])
    return RAGResult(answer=ans, chunk_ids=ids_used, context=ctx)


demo = naive_rag(AUTH_QUESTION)
print("Answer:", demo.answer[:200], "...")
print("Chunks used:", demo.chunk_ids)


---

## 12. Demonstration Queries

Run the full pipeline on three question types. After each call, verify that retrieved **chunk_ids** match the topic you expect.


In [ ]:
q_auth = "How should API clients send credentials?"
r_auth = naive_rag(q_auth, k=3)
print("Q:", q_auth)
print("Chunks:", r_auth.chunk_ids)
print("A:", r_auth.answer)
print()


In [ ]:
q_migrate = "What command applies Alembic migrations after pulling code?"
r_migrate = naive_rag(q_migrate, k=3)
print("Q:", q_migrate)
print("Chunks:", r_migrate.chunk_ids)
print("A:", r_migrate.answer)
print()


In [ ]:
q_test = "Where do I put shared pytest fixtures for database setup?"
r_test = naive_rag(q_test, k=3)
print("Q:", q_test)
print("Chunks:", r_test.chunk_ids)
print("A:", r_test.answer)


---

## 13. Module Structure Preview (FastAPI App)

Naive RAG in a FastAPI service typically splits into modules — even before adopting patterns from notebook **03**:

```
app/
├── main.py              # FastAPI app, POST /ask
├── config.py            # OPENAI_API_KEY, models, k
├── corpus/
│   └── documents.py     # DOCUMENTS or loader from files
├── rag/
│   ├── embedder.py      # embed_texts()
│   ├── index.py         # build_collection(), fresh_collection()
│   ├── retrieve.py      # retrieve()
│   ├── prompt.py        # build_context(), build_user_message()
│   └── pipeline.py      # naive_rag() -> RAGResult
└── schemas.py           # AskRequest, AskResponse
```

```
  HTTP POST /ask
       │
       ▼
  pipeline.naive_rag(question)
       │
       ├── retrieve.py  ──► Chroma
       ├── prompt.py    ──► context string
       └── OpenAI chat  ──► answer
```

Keeping **retrieve**, **prompt**, and **generate** in separate files lets you test retrieval without spending tokens on generation — the debugging split introduced in notebook **01**.


---

## 14. What Naive RAG Does Not Solve

Naive RAG is a **baseline**, not a finished product. The rest of this section builds on it:

| Gap | Symptom | See notebook |
|-----|---------|--------------|
| **Architecture patterns** | Monolith pipeline hard to scale/test | **03. RAG Architecture Patterns** |
| **Knowledge base / ingest** | PDFs, HTML, refresh cadence | **04. Knowledge Base and Ingest** |
| **Prompting / citations** | Weak refusals, no quote discipline | **05. Prompting and Context Injection** |
| **Context window limits** | Truncation, lost evidence | **06. Context Assembly and Window Management** |
| **Advanced retrieval** | Missed keywords, no reranking | **07. Advanced Retrieval** |
| **Failure modes / grounding** | Answer contradicts context | **08. RAG Failure Modes and Grounding** |
| **Evaluation** | No quality metrics | **09. Evaluating RAG End-to-End** |
| **Production** | Caching, observability, SLAs | **10. Production Patterns** |
| **Multi-turn** | Follow-up questions lose thread | **11. Conversational and Multi-Turn RAG** |
| **Agents / tools** | Live APIs, not static docs | **12. RAG vs Agents and Tool Use** |

Chunking and embedding model choice live in **05. Embeddings & Vector DBs** (**04**, **06**).


---

## 15. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Different embed model at query time | Random retrieval | Pin `EMBED_MODEL` everywhere |
| $k$ too large on small corpus | Noise drowns signal | Start $k=3$, inspect ids |
| No grounding instructions | Model ignores context | Strong system message + temperature 0 |
| Context in wrong message role | Hard-to-debug prompts | Log full message list |
| Trusting distance as probability | False confidence | Calibrate on your data |
| Skipping bare-LLM comparison | Cannot prove RAG value | A/B like section 9 |
| Reusing collection after doc edits | Stale vectors | `fresh_collection()` or versioned index |


---

## 16. Debugging Checklist

When answers look wrong, inspect **in order** — do not tweak the LLM first.

1. **Retrieval** — Are the expected chunk ids (e.g. **c3** for auth) in top-$k$? If not, fix embeddings, chunking, or $k$.
2. **Context** — Does `build_context` output contain the fact you need? Print it verbatim.
3. **Prompt** — Are system rules and user context actually sent? Log the full `messages` list.
4. **Generation** — Paste the gold chunk manually into the user message. If the model still fails, tune prompts or model — not retrieval.
5. **Usage** — Did `prompt_tokens` jump? You may be stuffing too much text (notebook **06**).

```
  wrong answer?
       │
       ▼
  chunk ids correct? ──no──► fix retrieve / index
       │
      yes
       ▼
  context contains fact? ──no──► fix k or chunking
       │
      yes
       ▼
  model ignores context? ──yes──► prompt / temperature
```


---

## 17. Summary

**Naive RAG** = **index → retrieve → augment → generate**. This notebook implemented that loop end-to-end:

- Indexed **c1–c5** into in-memory Chroma with **cosine** space and OpenAI embeddings
- Retrieved top-$k$ chunks, explained **distance vs similarity**, and discussed choosing $k$
- Built **system** and **user** messages with `temperature=0` for factual Q&A
- Compared **bare LLM** vs **RAG** answers and inspected **token usage**
- Wrapped the flow in `naive_rag()` returning a **`RAGResult`** dataclass

Back: **01. Introduction to RAG** — motivation and vocabulary.

Next: **03. RAG Architecture Patterns** — modular pipelines, workers, and scaling beyond this single-notebook script.
